# SR-MoE Training

**Sparse Routing Mixture-of-Experts** with popcode VQ codebook + ring attractor.

This notebook trains the SR-MoE routing architecture end-to-end on the associative
memory task. Run all cells in sequence.

## What is SR-MoE?

1. **Popcode Router** â€” replaces softmax with a learned VQ codebook. The router
   selects a discrete codeword (one of N codes), each of which is a distribution
   over the 3 routing actions. Closer to cortical sparse coding than MoE gating.

2. **Ring Attractor** â€” a 2D continuous attractor maintains routing state across
   steps. Routing decisions depend on both the current token and the ring phase,
   making routing path-dependent (vs stateless per-token softmax).

3. **Adversarial Diversity Regularizer** â€” penalizes code collapse (usage imbalance
   + low code separation) so all codes stay active.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy', 'wandb', '--quiet'], check=True)

import os, math, random, json, time
import torch
import torch.nn as nn
import torch.nn.functional as F

print('torch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
# ─── FRACTURED SRMoE HEAD ────────────────────────────────────────────────────
# Replaces the monolithic encoder with 4 independent shards, each processing
# a different subset of inputs. The budget shard gets dedicated capacity so
# budget dynamics can actually learn (was 1 column in a 141-dim concat — too thin).
#
# Architecture:
#   shard_hidden  (128 → 64 → 32): core hidden state
#   shard_budget  (4   → 32 → 16): budget + step features, SEPARATE pathway
#   shard_ring    (16  → 32 → 16): ring attractor projection
#   shard_cross   (132 → 48 → 24): hidden × budget interaction terms
#   fusion       (88  → 128 → n_codes): recombines all shards
#   shard_vote_net: per-shard confidence weighting (low entropy vote = trust)
# ─────────────────────────────────────────────────────────────────────────────

class FracturedSRMoEHead(nn.Module):
    def __init__(self, hidden_dim, n_codes=16, ring_dim=16, n_prefs=32,
                 tau_start=1.0, tau_end=0.01, anneal_steps=5000, lambda_div=0.1,
                 n_shards=4):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_codes = n_codes
        self.ring_dim = ring_dim
        self.n_prefs = n_prefs
        self._tau_start = tau_start
        self._tau_end = tau_end
        self.anneal_steps = anneal_steps
        self.n_shards = n_shards
        self._step = 0
        self._eval_mode = False

        self.ring = RingAttractor(hidden_dim, ring_dim, n_prefs)
        self.ring_state_proj = nn.Linear(n_prefs, ring_dim)

        # Shard 1: core hidden state
        self.shard_hidden = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.Tanh(),
            nn.Linear(64, 32))

        # Shard 2: budget + step features — THE key fix
        # Raw budget was 1 dim in 141-dim concat; now gets dedicated 2-layer net
        self.shard_budget = nn.Sequential(
            nn.Linear(4, 32), nn.Tanh(),   # 4 = budget_rem, step_pos, step_parity, is_query
            nn.Linear(32, 16))

        # Shard 3: ring attractor state
        self.shard_ring = nn.Sequential(
            nn.Linear(ring_dim, 32), nn.Tanh(),
            nn.Linear(32, 16))

        # Shard 4: hidden × budget cross-features
        # Captures interactions the fusion network can't easily learn from scratch
        self.shard_cross = nn.Sequential(
            nn.Linear(hidden_dim + 4, 48), nn.Tanh(),
            nn.Linear(48, 24))

        shard_emb_dim = 32 + 16 + 16 + 24  # = 88

        # Per-shard confidence: how much to trust each shard's embedding
        self.shard_vote_net = nn.Sequential(
            nn.Linear(shard_emb_dim, n_shards))

        # Fusion: combine shard embeddings → routing logits
        self.fusion = nn.Sequential(
            nn.Linear(shard_emb_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, n_codes))

        # Confidence projection: 32-dim weighted shard sum -> n_codes (16)
        self.confidence_proj = nn.Linear(32, n_codes)

        # Codebook: discrete routing actions
        cb = torch.zeros(n_codes, N_ACTIONS)
        for i in range(n_codes): cb[i, i % N_ACTIONS] = 1.0
        self.codebook = nn.Parameter(cb)

        self.div_reg = DiversityRegularizer(n_codes,
            lambda_usage=lambda_div*0.5, lambda_discrim=lambda_div*0.25)

    @property
    def tau(self):
        if self._eval_mode: return 0.0
        frac = min(self._step / max(self.anneal_steps, 1), 1.0)
        return self._tau_start + frac * (self._tau_end - self._tau_start)

    def set_eval_mode(self, hard=True): self._eval_mode = hard
    def reset(self): self._step = 0; self.ring.reset()

    def _extra(self, budget_remaining, step_pos, step_parity, is_query_phase, device):
        return torch.tensor([budget_remaining, step_pos, step_parity, is_query_phase],
                           dtype=torch.float, device=device)

    def forward(self, hidden, budget_remaining, step_pos, step_parity, is_query_phase):
        self._step += 1
        h = hidden.view(-1)
        device = h.device

        self.ring(h)
        rs = self.ring.get_state()
        rp = torch.tanh(self.ring_state_proj(rs))

        extra = self._extra(budget_remaining, step_pos, step_parity, is_query_phase, device)

        # Fractured encoding — each shard independent
        shard_h = self.shard_hidden(h)
        shard_b = self.shard_budget(extra.unsqueeze(0)).squeeze(0)
        shard_r = self.shard_ring(rp.unsqueeze(0)).squeeze(0)
        shard_c = self.shard_cross(torch.cat([h, extra], dim=-1).unsqueeze(0)).squeeze(0)

        shard_emb = torch.cat([shard_h, shard_b, shard_r, shard_c], dim=-1)  # [88]

        # Per-shard confidence weights
        vote_logits = self.shard_vote_net(shard_emb.unsqueeze(0)).squeeze(0)
        vote_probs = F.softmax(vote_logits, dim=-1)  # sum to 1
        vote_entropy = -(vote_probs * (vote_probs + 1e-9).log()).sum()
        vote_confidence = 1.0 - vote_entropy / math.log(self.n_shards)

        # Fusion → routing logits
        pre_code = self.fusion(shard_emb.unsqueeze(0)).squeeze(0)

        # Confidence-weighted bias: weighted sum of shard embeddings, projected to n_codes
        # vote_probs[i] is a scalar weight per shard. vote_confidence scales the whole thing.
        # Each shard contributes its weighted embedding to a shared bias, then we project to n_codes.
        max_shard_dim = 32
        bias = torch.zeros(max_shard_dim, device=device)
        bias[:shard_h.shape[0]] += vote_probs[0] * shard_h
        bias[:shard_b.shape[0]] += vote_probs[1] * shard_b
        bias[:shard_r.shape[0]] += vote_probs[2] * shard_r
        bias[:shard_c.shape[0]] += vote_probs[3] * shard_c
        # Project from max_shard_dim (32) down to n_codes (16) so it broadcasts with pre_code
        confidence_bias = vote_confidence.detach() * self.confidence_proj(bias)
        pre_code = pre_code + 0.1 * confidence_bias

        # Code selection
        if self._eval_mode or self.tau <= 0:
            code_idx = pre_code.argmax().item()
        else:
            gumbel = F.gumbel_softmax(pre_code.unsqueeze(0), tau=self.tau, hard=True)
            code_idx = int(gumbel.argmax().item())

        sel = F.embedding(torch.tensor([code_idx], device=device), self.codebook).squeeze(0)
        action_logits = sel + (self.codebook[code_idx] - self.codebook[code_idx]).detach()
        action_idx = int(action_logits.argmax().item())

        probs = F.softmax(action_logits, dim=-1)
        entropy = -(probs * (probs + 1e-9).log()).sum().item()
        salience_entropy = entropy / math.log(N_ACTIONS)

        return action_logits, action_idx, salience_entropy

    def diversity_loss(self, hidden, budget_remaining):
        h = hidden.view(-1)
        device = h.device
        rs = self.ring.get_state()
        rp = torch.tanh(self.ring_state_proj(rs))
        extra = self._extra(budget_remaining, 0.0, 0.0, 0.0, device)

        shard_h = self.shard_hidden(h)
        shard_b = self.shard_budget(extra.unsqueeze(0)).squeeze(0)
        shard_r = self.shard_ring(rp.unsqueeze(0)).squeeze(0)
        shard_c = self.shard_cross(torch.cat([h, extra], dim=-1).unsqueeze(0)).squeeze(0)
        shard_emb = torch.cat([shard_h, shard_b, shard_r, shard_c], dim=-1)
        pre_code = self.fusion(shard_emb.unsqueeze(0)).squeeze(0)

        code_probs = F.softmax(pre_code, dim=-1)
        code_idx = code_probs.argmax().item()
        return self.div_reg(self.codebook, pre_code.unsqueeze(0),
                          torch.tensor([code_idx], device=device))


# Alias so ArchitectureB / trainer pick up the fractured head automatically
SRMoEHead = FracturedSRMoEHead
print('FracturedSRMoEHead (4 shards) loaded — SRMoEHead aliased to it')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# DAgger: Dataset Aggregation for Phase 2
#
# Problem: Phase 1 trains router on oracle-routed episodes (optimal memory,
# optimal budget). Phase 2 runs the learned policy, which visits DIFFERENT
# states — different memory content, different budget levels. The oracle
# never saw those states. Classic distributional shift.
#
# DAgger fix: run the current policy, query the oracle at each step,
# collect (state, oracle_action) pairs. Train on both DAgger data AND
# fresh oracle episodes. The training distribution now matches inference.
#
# Safety: episodes with blocked_frac > 0.6 are discarded — the policy
# routed so badly the episode is unrecoverable. Training on those states
# would accelerate collapse, not fix it.
# ═══════════════════════════════════════════════════════════════════════

class DAggerReplayBuffer:
    """
    FIFO replay buffer for DAgger (state, oracle_action) transitions.
    State = (hidden, budget_norm, step_pos, step_parity, is_query_phase)
    Only stores tensors that are safe to clone (hidden is 1D). Ring state
    is NOT stored — it is deterministically reconstructable from hidden
    via the ring attractor's input projection, so recomputing it is free.
    """
    def __init__(self, max_size=5000):
        self.max_size = max_size
        self.buffer = []

    def append(self, hidden, budget_norm, step_pos, step_parity, is_query, oracle_action_idx):
        entry = {
            'hidden': hidden.detach().clone(),
            'budget_norm': budget_norm,
            'step_pos': step_pos,
            'step_parity': step_parity,
            'is_query': is_query,
            'oracle_action': oracle_action_idx,
        }
        self.buffer.append(entry)
        if len(self.buffer) > self.max_size:
            self.buffer.pop(0)

    def sample(self, batch_size, device):
        if len(self.buffer) == 0:
            return None
        indices = random.sample(range(len(self.buffer)), min(batch_size, len(self.buffer)))
        hiddens     = torch.stack([self.buffer[i]['hidden'] for i in indices])
        budget_norms= torch.tensor([self.buffer[i]['budget_norm'] for i in indices], dtype=torch.float, device=device)
        step_pos    = torch.tensor([self.buffer[i]['step_pos'] for i in indices], dtype=torch.float, device=device)
        step_parity = torch.tensor([self.buffer[i]['step_parity'] for i in indices], dtype=torch.float, device=device)
        is_query    = torch.tensor([self.buffer[i]['is_query'] for i in indices], dtype=torch.float, device=device)
        oracle_acts = torch.tensor([self.buffer[i]['oracle_action'] for i in indices], dtype=torch.long, device=device)
        return hiddens, budget_norms, step_pos, step_parity, is_query, oracle_acts

    def __len__(self): return len(self.buffer)


def dagger_collection_episode(model, episode, buffer, device, max_blocked_frac=0.6):
    """
    Run one episode with the CURRENT (partially learned) policy, collect
    (state, oracle_action) pairs for DAgger.
    
    This does NOT modify model state permanently — ring state is saved/restored
    around each oracle query so the trajectory is unaffected by the oracle call.
    
    Returns: dict of metrics, or None if episode is too corrupted (blocked > threshold).
    """
    m = model.eval()
    m.reset_episode()

    # Temporary storage for this episode's transitions
    episode_hidden = []
    episode_budget = []
    episode_step_pos = []
    episode_step_parity = []
    episode_is_query = []
    episode_oracle_act = []

    skip_cnt = read_cnt = write_cnt = blocked_cnt = 0
    encode_occupancies = []

    with torch.no_grad():
        for t in range(len(episode.input_ids)):
            tok = episode.input_ids[t].to(device)
            is_query_step = (t >= QUERY_START)
            budget_norm = m.budget.budget_remaining_normalised

            # ── Current policy forward pass ──────────────────────────────
            logits, info = m(tok)
            ai = info['action_idx']
            
            if ai == 0: skip_cnt += 1
            elif ai == 1: read_cnt += 1
            elif ai == 2: write_cnt += 1
            if info.get('blocked'): blocked_cnt += 1

            if t < QUERY_START:
                encode_occupancies.append(info['occupancy'])

            # ── Oracle query (no side effects) ─────────────────────────
            # Save ring state before oracle call, restore after
            saved_ring_state = m.routing.ring._ring_state.clone() if m.routing.ring._ring_state is not None else None
            oracle_act_idx = _O2I.get(episode.oracle_actions[t], 0)
            # Restore ring state — oracle query has zero side effects on trajectory
            if saved_ring_state is not None:
                m.routing.ring._ring_state = saved_ring_state

            # ── Collect transition ──────────────────────────────────────
            # Clone hidden to break graph (no gradient needed for DAgger collection)
            episode_hidden.append(m._hidden.detach().clone())
            episode_budget.append(budget_norm)
            episode_step_pos.append(t / EPISODE_LENGTH)
            episode_step_parity.append(float(t % 2))
            episode_is_query.append(float(is_query_step))
            episode_oracle_act.append(oracle_act_idx)

    n_steps = len(episode.input_ids)
    blocked_frac = blocked_cnt / max(n_steps, 1)

    # Safety valve: discard episodes that are too corrupted
    if blocked_frac > max_blocked_frac:
        return None  # caller should skip adding to buffer

    # ── Add episode transitions to buffer ─────────────────────────────
    for i in range(len(episode_hidden)):
        buffer.append(
            episode_hidden[i],
            episode_budget[i],
            episode_step_pos[i],
            episode_step_parity[i],
            episode_is_query[i],
            episode_oracle_act[i]
        )

    return {
        'blocked_frac': blocked_frac,
        'skip_frac': skip_cnt / n_steps,
        'read_frac': read_cnt / n_steps,
        'write_frac': write_cnt / n_steps,
        'avg_occupancy': sum(encode_occupancies) / max(len(encode_occupancies), 1),
        'buffer_size': len(buffer),
    }


def dagger_loss_from_buffer(model, buffer, batch_size, device):
    """
    Compute DAgger loss from the replay buffer.
    Samples batch_size transitions, runs them through the router (with gradient),
    and computes CE against the oracle actions stored in the buffer.
    
    This trains the router on states visited by the CURRENT policy but with
    oracle labels — bridging the distribution gap between oracle and learned routing.
    """
    batch = buffer.sample(batch_size, device)
    if batch is None:
        return torch.tensor(0.0, device=device), {}

    hiddens, budget_norms, step_pos, step_parity, is_query, oracle_acts = batch
    
    total_loss = torch.tensor(0.0, device=device)
    n = hiddens.shape[0]

    # Route stop_grad = False — we WANT gradients here (this IS the training)
    for i in range(n):
        h = hiddens[i]  # [hidden_dim]
        bn = budget_norms[i].item()
        sp = step_pos[i].item()
        sq = step_parity[i].item()
        iq = is_query[i].item()

        # Forward through router — gradient flows back to router params
        # We temporarily set the ring state from hidden (deterministic, no side effects)
        m = model
        saved_ring = m.routing.ring._ring_state.clone() if m.routing.ring._ring_state is not None else None
        m.routing.ring._ring_state = None  # force ring to recompute from hidden

        router_logits, action_idx, salience = m.routing(
            h, bn, sp, sq, iq
        )

        # Restore ring state
        m.routing.ring._ring_state = saved_ring

        oracle_action = oracle_acts[i]
        if oracle_action.item() != IGNORE_INDEX:
            step_loss = F.cross_entropy(
                router_logits.unsqueeze(0),
                oracle_action.unsqueeze(0)
            )
            total_loss = total_loss + step_loss

    avg_loss = total_loss / max(n, 1)
    return avg_loss, {'dagger_n_samples': n}


print('DAgger infrastructure loaded')


In [ ]:
# Constants
N_ACTIONS = 3   # SKIP=0, READ=1, WRITE=2
ORACLE_TO_ACTION_IDX = {'SKIP': 0, 'READ': 1, 'WRITE': 2, 'PERCEIVE': 0}
IGNORE_INDEX = -100

# Ring Attractor
class RingAttractor(nn.Module):
    def __init__(self, hidden_dim, ring_dim=16, n_prefs=32, dt=0.5, decay=0.95):
        super().__init__()
        self.hidden_dim = hidden_dim; self.ring_dim = ring_dim
        self.n_prefs = n_prefs; self.dt = dt; self.decay = decay

        self.register_buffer('_prefs',
            torch.linspace(0, 2*math.pi, n_prefs, dtype=torch.float32))
        self.input_proj = nn.Sequential(
            nn.Linear(hidden_dim, ring_dim*2), nn.Tanh())
        self.lateral_w = nn.Parameter(torch.zeros(n_prefs, n_prefs))
        self._init_lateral_weights()
        self._ring_state = None

    def _init_lateral_weights(self):
        with torch.no_grad():
            d = self._prefs.unsqueeze(1) - self._prefs.unsqueeze(0)
            d = torch.atan2(torch.sin(d), torch.cos(d))
            exc = torch.exp(-0.5*(d/2.0)**2)
            inh = 0.3*torch.exp(-0.5*(d/5.0)**2)
            w = (exc - inh) / (exc - inh).abs().mean() * 0.1
            self.lateral_w.copy_(w)

    def reset(self): self._ring_state = None

    def forward(self, hidden):
        is_1d = hidden.dim() == 1
        if is_1d: hidden = hidden.unsqueeze(0)
        batch_size = hidden.shape[0]
        ri = self.input_proj(hidden)
        theta = torch.atan2(ri[..., 1], ri[..., 0])
        prev = self._ring_state
        if prev is None:
            prev = torch.zeros(self.n_prefs, device=hidden.device)
        prev_2d = prev.unsqueeze(0)
        drive = F.relu(torch.cos(self._prefs.unsqueeze(0) - theta.unsqueeze(1)))
        lateral = (self.lateral_w @ prev).unsqueeze(0).expand(batch_size, -1)
        new_state = prev_2d + self.dt * (drive + 0.3*lateral - prev_2d - self.decay*prev_2d)
        new_state = F.relu(new_state)
        new_state = new_state / (new_state.sum(dim=-1, keepdim=True) + 1e-8)
        self._ring_state = new_state.squeeze(0).detach()
        return self._ring_state

    def get_state(self):
        return self._ring_state if self._ring_state is not None else \
               torch.zeros(self.n_prefs, device=next(self.parameters()).device)

# Diversity Regularizer
class DiversityRegularizer(nn.Module):
    def __init__(self, n_codes, lambda_usage=0.05, lambda_discrim=0.025):
        super().__init__()
        self.n_codes = n_codes
        self.lambda_usage = lambda_usage
        self.lambda_discrim = lambda_discrim

    def forward(self, codebook, pre_code_logits, code_indices):
        device = codebook.device
        batch_size = code_indices.shape[0]
        usage_hist = torch.zeros(self.n_codes, device=device)
        for idx in code_indices: usage_hist[idx] += 1
        usage_p = usage_hist / (batch_size + 1e-8)
        usage_entropy = -(usage_p * (usage_p + 1e-8).log()).sum()
        usage_loss = 1.0 - usage_entropy / (math.log(self.n_codes) + 1e-8)
        cb_probs = F.softmax(codebook, dim=-1)
        discrimin_loss = torch.tensor(0.0, device=device)
        n_pairs = 0
        for i in range(self.n_codes):
            for j in range(i+1, self.n_codes):
                kl = F.kl_div(cb_probs[i].log().unsqueeze(0),
                              cb_probs[j].unsqueeze(0), reduction='batchmean')
                discrimin_loss += F.relu(0.5 - kl)
                n_pairs += 1
        if n_pairs > 0: discrimin_loss /= n_pairs
        loss = self.lambda_usage * usage_loss + self.lambda_discrim * discrimin_loss
        return loss, {'usage': usage_loss.item(), 'discrim': discrimin_loss.item()}

# SR-MoE Head

# SRMoEHead moved to FracturedSRMoEHead (see cell 2)



In [ ]:
# Architecture B (simplified)
V_KEY_VAL = 50; V_FILLER = 10; V_TOTAL = V_KEY_VAL + V_FILLER
OUTPUT_VOCAB = V_KEY_VAL
EPISODE_LENGTH = 190; QUERY_START = 150

class FrozenEmbedding(nn.Module):
    def __init__(self, v_total=V_TOTAL, d_embed=64, seed=0):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.register_buffer('weight', F.normalize(torch.randn(v_total, d_embed, generator=g), dim=-1))
    def forward(self, token_ids): return F.embedding(token_ids, self.weight)

class BudgetEnforcer:
    def __init__(self, B=20, T_window=100):
        self.B, self.T_window = B, T_window; self._reset()
    def _reset(self):
        self.accesses = []; self.total_permitted = 0
        self.blocked = 0; self.requested = 0
    def step(self, action_idx):
        self.requested += 1
        if action_idx == 0: return True
        window = self.accesses[max(0, len(self.accesses)-self.T_window):]
        if len(window) >= self.B: self.blocked += 1; return False
        self.accesses.append(action_idx); self.total_permitted += 1; return True
    @property
    def budget_remaining(self):
        window = self.accesses[max(0, len(self.accesses)-self.T_window):]
        return max(0, self.B - len(window))
    @property
    def budget_remaining_normalised(self): return self.budget_remaining / self.B

class ExternalMemory(nn.Module):
    def __init__(self, capacity=200, key_dim=64, value_dim=64, write_threshold=0.5):
        super().__init__()
        self.capacity = capacity; self.key_dim = key_dim
        self.value_dim = value_dim; self.write_threshold = write_threshold
        self.reset()
    def reset(self): self.keys = []; self.values = []; self.saliences = []
    def write(self, key, value, salience=1.0, override_threshold=False):
        if salience < self.write_threshold and not override_threshold: return
        if len(self.keys) >= self.capacity:
            idx = self.saliences.index(min(self.saliences))
            self.keys.pop(idx); self.values.pop(idx); self.saliences.pop(idx)
        self.keys.append(key.detach()); self.values.append(value.detach())
        self.saliences.append(salience)
    def read(self, query):
        if not self.keys:
            return torch.zeros(self.value_dim, device=query.device), torch.tensor(0.0)
        keys_t = torch.stack([k.squeeze(0) if k.dim()>1 else k for k in self.keys])
        scores = F.cosine_similarity(query.unsqueeze(0), keys_t, dim=-1)
        idx = scores.argmax().item()
        return self.values[idx].to(query.device), scores[idx]
    @property
    def occupancy(self): return len(self.keys)

class ArchitectureB(nn.Module):
    def __init__(self, d_embed=64, hidden_dim=128, budget_B=55, budget_T=190,
                 write_threshold=0.5, embed_seed=0, use_sr_moe=True,
                 n_codes=16, ring_dim=16, n_prefs=32, lambda_div=0.1,
                 tau_start=1.0, tau_end=0.01, anneal_steps=5000):
        super().__init__()
        self.d_embed = d_embed; self.hidden_dim = hidden_dim
        self.budget_B = budget_B; self.budget_T = budget_T
        self.write_threshold = write_threshold; self.use_sr_moe = use_sr_moe

        self.embedding = FrozenEmbedding(V_TOTAL, d_embed, seed=embed_seed)
        self.gru = nn.GRUCell(d_embed + d_embed, hidden_dim)

        if use_sr_moe:
            self.routing = SRMoEHead(hidden_dim, n_codes=n_codes, ring_dim=ring_dim,
                                     n_prefs=n_prefs, tau_start=tau_start, tau_end=tau_end,
                                     anneal_steps=anneal_steps, lambda_div=lambda_div)
        else:
            self.routing = nn.Sequential(
                nn.Linear(hidden_dim+4, hidden_dim), nn.ReLU(),
                nn.Linear(hidden_dim, 3))

        self.memory = ExternalMemory(200, d_embed, d_embed, write_threshold)
        self.budget = BudgetEnforcer(budget_B, budget_T)
        self.output_head = nn.Linear(hidden_dim + d_embed, OUTPUT_VOCAB)

        self._hidden = None; self._step = 0
        self._pending_write_key = None; self._prev_emb = None
        self._primed_oracle_actions = None

    def reset_episode(self):
        self._hidden = torch.zeros(1, self.hidden_dim, device=next(self.parameters()).device)
        self._step = 0; self._pending_write_key = None; self._prev_emb = None
        self.memory.reset(); self.budget._reset()
        if hasattr(self.routing, 'reset'): self.routing.reset()

    def prime_oracle_schedule(self, actions): self._primed_oracle_actions = actions

    def forward(self, token_id, route_stop_grad=True, ablation_no_mem=False,
                ablation_rand_routing=False):
        if self._hidden is None: self.reset_episode()
        device = token_id.device
        t = self._step; self._step += 1
        is_query_step = (t >= QUERY_START)

        emb = self.embedding(token_id.view(1) if token_id.dim()==0 else token_id).squeeze(0)
        budget_norm = self.budget.budget_remaining_normalised
        _O2I = {'SKIP': 0, 'READ': 1, 'WRITE': 2, 'PERCEIVE': 3}

        if self._primed_oracle_actions and t < len(self._primed_oracle_actions):
            action_idx = _O2I.get(self._primed_oracle_actions[t], 0)
            hfr = self._hidden  # oracle routing: full gradient
            lr, _, ls = self.routing(hfr, budget_norm, t/EPISODE_LENGTH, float(t%2), float(is_query_step))
        elif ablation_rand_routing:
            action_idx = random.randint(0, 2)
            lr = torch.zeros(3, device=device); ls = 1.0/3
        else:
            hfr = self._hidden  # FIX #1: router gets full task gradient always (was detaching and killing Phase 2 learning)
            lr, action_idx, ls = self.routing(hfr, budget_norm, t/EPISODE_LENGTH, float(t%2), float(is_query_step))

        if action_idx == 3:
            self._pending_write_key = emb.detach().clone(); action_idx = 0

        _act_before = action_idx
        permitted = self.budget.step(0 if action_idx==3 else action_idx)  # PERCEIVE(3) is no-op on budget
        if not permitted: action_idx = 0

        retrieved = torch.zeros(self.d_embed, device=device)
        if action_idx == 1 and not ablation_no_mem:
            retrieved, _ = self.memory.read(emb); retrieved = retrieved.to(device)
        elif action_idx == 2 and not ablation_no_mem:
            wk = self._pending_write_key if self._pending_write_key is not None \
                 else (self._prev_emb if self._prev_emb is not None else emb)
            ws = ls if self._primed_oracle_actions else 1.0
            self.memory.write(wk, emb, salience=ws); self._pending_write_key = None

        self._hidden = self.gru(torch.cat([emb, retrieved], dim=-1).unsqueeze(0), self._hidden)
        self._prev_emb = emb.detach().clone()
        logits = self.output_head(torch.cat([self._hidden.squeeze(0), retrieved], dim=-1))

        info = {
            'step': t,
            'action': {0:'SKIP',1:'READ',2:'WRITE'}.get(action_idx,'SKIP'),
            'action_idx': action_idx,
            'blocked': (not permitted) and _act_before in (1,2),
            'permitted': permitted,
            'salience_entropy': ls,
            'budget_remaining': self.budget.budget_remaining,
            'occupancy': self.memory.occupancy,
            'routing_logits': lr}
        return logits, info

print('ArchitectureB loaded OK')


In [ ]:
from dataclasses import dataclass
from typing import List

N_GENUINE = 20; EPISODE_LENGTH = 190; QUERY_START = 150
IGNORE_INDEX = -100

@dataclass
class Episode:
    input_ids: torch.Tensor
    oracle_actions: List[str]
    target_value_ids: torch.Tensor
    genuine_keys: List[int] = None
    genuine_values: List[int] = None

def generate_episode(seed=0):
    rng = random.Random(seed)
    gk = [rng.randint(0, 49) for _ in range(N_GENUINE)]
    gv = [rng.randint(0, 49) for _ in range(N_GENUINE)]
    fk = [rng.randint(0, 49) for _ in range(60)]
    fv = [rng.randint(0, 49) for _ in range(60)]

    inp, ora = [], []
    # Encode: 100 steps (50 key+value pairs)
    for i in range(100):
        if i % 2 == 0:
            inp.append(gk[i//2] if i//2 < N_GENUINE else fk[i//2])
            ora.append('PERCEIVE' if i//2 < N_GENUINE else 'SKIP')
        else:
            inp.append(gv[i//2] if i//2 < N_GENUINE else fv[i//2])
            ora.append('WRITE' if i//2 < N_GENUINE else 'SKIP')
    # Delay: 50 steps
    for _ in range(50):
        inp.append(rng.randint(0, 49)); ora.append('SKIP')
    # Query: 20 steps
    tv = [IGNORE_INDEX] * EPISODE_LENGTH
    for qi, qk in enumerate(gk):
        inp.append(qk); inp.append(rng.randint(0, 49))
        ora.extend(['READ', 'SKIP'])
        tv[150 + qi*2] = gv[qi]

    return Episode(
        input_ids=torch.tensor(inp, dtype=torch.long),
        oracle_actions=ora,
        target_value_ids=torch.tensor(tv, dtype=torch.long),
        genuine_keys=gk, genuine_values=gv)

ep = generate_episode(0)
print(f'Episode: {len(ep.input_ids)} tokens, {N_GENUINE} key-value pairs')
print(f'Sample oracle: {ep.oracle_actions[:8]}')


In [ ]:
import wandb

class SRMoETrainer:
    def __init__(self, model, lr=3e-3, lambda_route=1.0, lambda_div=0.1,
                 lambda_margin=1.0, margin_target=0.5,
                 route_class_weights=[1.0, 10.0, 15.0]):
        self.model = model.to(DEVICE)
        self.lr = lr; self.lambda_route = lambda_route
        self.lambda_div = lambda_div; self.lambda_margin = lambda_margin
        self.margin_target = margin_target
        self.route_class_weights = route_class_weights
        self.lambda_density = 0.3  # FIX #4: penalize route density below threshold
        self.params = (list(model.routing.parameters())
            + list(model.output_head.parameters())
            + list(model.gru.parameters()))  # FIX #2: GRU in trainable params so hidden carries task signal
        self.opt = torch.optim.Adam(self.params, lr=lr)
        self.margin_loss_fn = torch.nn.MarginRankingLoss(margin=margin_target)
        self._wandb_run = wandb.run if hasattr(wandb, 'run') and wandb.run is not None else None
        self._ep_count = 0

    def _margin_loss(self, write_logits, skip_logits):
        pos = write_logits.unsqueeze(-1); neg = skip_logits.unsqueeze(0)
        return self.margin_loss_fn(pos, neg, torch.ones_like(pos))



    def _density_loss(self, action_idx_list):
        """FIX #4: penalize models that route fewer than 20% of steps to non-SKIP."""
        non_skip = sum(1 for a in action_idx_list if a != 0)
        frac = non_skip / max(len(action_idx_list), 1)
        target = 0.20
        if frac < 0.15:
            deficit = target - frac
            return self.lambda_density * (deficit ** 2)
        return None
    def train_episode(self, episode, use_oracle_routing=True):
        m = self.model; m.train(); m.reset_episode()
        m.prime_oracle_schedule(list(episode.oracle_actions) if use_oracle_routing else None)

        r_logits_list, oracle_labels = [], []
        answer_logits, answer_targets = [], []
        action_idx_list = []      # local list for route CE (FIX #5)
        self._action_idx_list = []  # track for density loss (FIX #4)
        # per-step metric accumulators
        skip_cnt = read_cnt = write_cnt = blocked_cnt = 0
        encode_occupancies = []
        encode_saliences = []

        for t in range(len(episode.input_ids)):
            tok = episode.input_ids[t].to(DEVICE)
            tgt = episode.target_value_ids[t].item()
            logits, info = m(tok)

            r_logits_list.append(info['routing_logits'])
            oa = episode.oracle_actions[t]
            oracle_labels.append(ORACLE_TO_ACTION_IDX.get(oa, IGNORE_INDEX))

            ai = info['action_idx']
            action_idx_list.append(ai)       # route density tracking
            self._action_idx_list.append(ai)  # FIX #4: for density loss
            if ai == 0: skip_cnt += 1
            elif ai == 1: read_cnt += 1
            elif ai == 2: write_cnt += 1
            if info.get('blocked'): blocked_cnt += 1
            if t < QUERY_START:
                encode_occupancies.append(info['occupancy'])
                encode_saliences.append(info['salience_entropy'])

            if tgt != IGNORE_INDEX:
                answer_logits.append(logits); answer_targets.append(tgt)

        n_steps = len(episode.input_ids)
        skip_frac = skip_cnt / n_steps
        read_frac = read_cnt / n_steps
        write_frac = write_cnt / n_steps
        blocked_frac = blocked_cnt / max(n_steps, 1)
        avg_occupancy = sum(encode_occupancies) / max(len(encode_occupancies), 1)
        avg_salience = sum(encode_saliences) / max(len(encode_saliences), 1)

        r_t = torch.stack(r_logits_list)
        # FIX #5: mask out encode-phase SKIP labels from route CE
        # Oracle SKIP during encode is a crutch, not a real routing decision
        r_lbl = torch.tensor(oracle_labels, dtype=torch.long, device=DEVICE)
        route_mask = torch.ones_like(r_lbl, dtype=torch.bool)  # default: include
        for ti, oa in enumerate(oracle_labels):
            if oa == IGNORE_INDEX:
                route_mask[ti] = False
            elif oa == 0 and ti < QUERY_START:  # SKIP during encode — skip this label
                route_mask[ti] = False
        if route_mask.sum() > 0:
            masked_r_t = r_t[route_mask]
            masked_r_lbl = r_lbl[route_mask]
            r_wt = torch.tensor(self.route_class_weights, dtype=torch.float, device=DEVICE)
        L_route = F.cross_entropy(r_t, r_lbl, weight=r_wt, ignore_index=IGNORE_INDEX)

        L_answer = torch.tensor(0.0, device=DEVICE); accuracy = 0.0
        if answer_logits:
            a_t = torch.stack(answer_logits)
            a_l = torch.tensor(answer_targets, dtype=torch.long, device=DEVICE)
            L_answer = F.cross_entropy(a_t, a_l)
            accuracy = (a_t.argmax(-1)==a_l).float().mean().item()

        L_margin = torch.tensor(0.0, device=DEVICE)
        if self.lambda_margin > 0 and len(r_logits_list) > 0:
            all_r = torch.stack(r_logits_list)
            L_margin = self._margin_loss(all_r[:,2], all_r[:,0])

        L_div = torch.tensor(0.0, device=DEVICE); div_comp = {}
        if self.lambda_div > 0 and hasattr(m.routing, 'diversity_loss'):
            L_div, div_comp = m.routing.diversity_loss(
                m._hidden.detach(), m.budget.budget_remaining_normalised)

        L_density = self._density_loss(self._action_idx_list)  # FIX #4
        loss = L_answer + self.lambda_route*L_route + self.lambda_margin*L_margin + self.lambda_div*L_div + (L_density if L_density is not None else torch.tensor(0.0, device=DEVICE))

        self.opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(self.params, 1.0); self.opt.step()

        # wandb logging
        if self._wandb_run is not None:
            self._wandb_run.log({
                'train/loss': loss.item(),
                'train/answer_loss': L_answer.item(),
                'train/route_loss': L_route.item(),
                'train/margin_loss': L_margin.item(),
                'train/div_loss': L_div.item(),
                'train/accuracy': accuracy,
                'train/skip_frac': skip_frac,
                'train/read_frac': read_frac,
                'train/write_frac': write_frac,
                'train/blocked_frac': blocked_frac,
                'train/avg_encode_occupancy': avg_occupancy,
                'train/avg_salience_entropy': avg_salience,
                'train/tau': m.routing.tau if hasattr(m.routing, 'tau') else 0.0,
                'train/div_usage': div_comp.get('usage', 0.0),
                'train/div_discrim': div_comp.get('discrim', 0.0),
            }, step=self._ep_count)
            self._ep_count += 1

        return {
            'loss': loss.item(), 'answer_loss': L_answer.item(),
            'route_loss': L_route.item(), 'margin_loss': L_margin.item(),
            'div_loss': L_div.item(), 'accuracy': accuracy,
            'skip_frac': skip_frac, 'read_frac': read_frac, 'write_frac': write_frac,
            'blocked_frac': blocked_frac, 'avg_occupancy': avg_occupancy,
            'avg_salience': avg_salience, 'div_components': div_comp,
        }

    def eval_free(self, episodes, hard=True):
        m = self.model.eval()
        correct, total = 0, 0
        write_margins, skip_fracs, read_fracs, write_fracs = [], [], [], []
        encode_occupancies = []
        with torch.no_grad():
            for ep in episodes:
                m.reset_episode()
                if hasattr(m.routing, 'set_eval_mode'): m.routing.set_eval_mode(hard)
                for t in range(len(ep.input_ids)):
                    tok = ep.input_ids[t].to(DEVICE)
                    tgt = ep.target_value_ids[t].item()
                    logits, info = m(tok, route_stop_grad=True)
                    if t >= QUERY_START and tgt != IGNORE_INDEX:
                        correct += int(logits.argmax().item() == tgt); total += 1
                    ai = info['action_idx']
                    r = info.get('routing_logits')
                    if r is not None and r.shape[-1] >= 3:
                        write_margins.append((r[2]-r[0]).item())
                    skip_fracs.append(1.0 if ai==0 else 0.0)
                    read_fracs.append(1.0 if ai==1 else 0.0)
                    write_fracs.append(1.0 if ai==2 else 0.0)
                    if t < QUERY_START:
                        encode_occupancies.append(info['occupancy'])
                if hasattr(m.routing, 'set_eval_mode') and not hard:
                    m.routing.set_eval_mode(False)
        n = max(len(write_margins), 1)
        return {
            'accuracy': correct/max(total,1),
            'write_margin': sum(write_margins)/n,
            'skip_frac': sum(skip_fracs)/n,
            'read_frac': sum(read_fracs)/n,
            'write_frac': sum(write_fracs)/n,
            'avg_encode_occupancy': sum(encode_occupancies)/max(len(encode_occupancies),1),
        }

print('Trainer class OK')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# DAgger training mixin: adds dagger_train_episode to SRMoETrainer
#
# Phase 2 now alternates between:
#   1. DAgger collection: run current policy, populate replay buffer
#   2. DAgger training: train on buffer samples + fresh oracle episodes
#
# beta schedule: start at 0.8 (mostly oracle), decay to 0.2 over DECAY_EPS
# ═══════════════════════════════════════════════════════════════════════

DAggerBuffer = DAggerReplayBuffer(max_size=5000)

DECAY_EPS = 500       # episodes over which beta decays from 0.8 to 0.2
INIT_BETA = 0.8       # start: 80% oracle actions, 20% policy states
FINAL_BETA = 0.2      # end: 20% oracle actions, 80% policy states
DAGGER_COLLECT_EVERY = 3   # collect DAgger every N training steps
DAGGER_BATCH_SIZE = 64     # DAgger samples per training step
ORACLE_EPISODES_PER_STEP = 1  # fresh oracle episodes per step

def _beta_schedule(ep_in_phase2):
    """Linear decay from INIT_BETA to FINAL_BETA over DECAY_EPS."""
    frac = min(ep_in_phase2 / max(DECAY_EPS - 1, 1), 1.0)
    return INIT_BETA + frac * (FINAL_BETA - INIT_BETA)


def dagger_train_step(trainer, dagger_buffer, beta, device, collect_stats=None):
    """
    One DAgger training step:
      - Sample DAGGER_BATCH_SIZE transitions from buffer, compute dagger loss
      - Generate ORACLE_EPISODES_PER_STEP fresh oracle episodes, compute oracle loss
      - Mix: dagger_loss*(1-beta) + oracle_loss*beta, single backward pass

    CRITICAL: model must be in train() mode so autograd works.
    dagger_collection_episode sets model to eval() — switch back here.
    Also: we DON'T call trainer.train_episode (it does its own backward internally).
    Instead run the forward pass manually and accumulate a single L_oracle tensor.
    """
    # Force train mode — dagger_collection_episode leaves model in eval() mode
    trainer.model.train()

    # DAgger loss from buffer: forward pass through router with gradient
    L_dagger, d_info = dagger_loss_from_buffer(
        trainer.model, dagger_buffer, DAGGER_BATCH_SIZE, device
    )

    # Oracle loss: run forward manually (NOT via train_episode — it calls backward)
    L_oracle_total = torch.tensor(0.0, device=device)
    oracle_acc = 0.0
    for _ in range(ORACLE_EPISODES_PER_STEP):
        ep = generate_episode(int(random.random() * 1e6))
        ep_stats = _run_oracle_episode(trainer.model, ep, device)
        L_oracle_total = L_oracle_total + ep_stats['loss']
        oracle_acc = ep_stats.get('accuracy', 0.0)
    L_oracle = L_oracle_total / ORACLE_EPISODES_PER_STEP

    # Mix losses and single backward pass
    L_mixed = (1 - beta) * L_dagger + beta * L_oracle

    trainer.opt.zero_grad()
    L_mixed.backward()
    torch.nn.utils.clip_grad_norm_(trainer.params, 1.0)
    trainer.opt.step()

    if collect_stats is not None:
        collect_stats['dagger_loss'] = L_dagger.item()
        collect_stats['oracle_loss'] = L_oracle.item()
        collect_stats['mixed_loss'] = L_mixed.item()
        collect_stats['beta'] = beta
        collect_stats['buffer_size'] = len(dagger_buffer)
        collect_stats['dagger_samples'] = d_info.get('dagger_n_samples', 0)
        collect_stats['oracle_acc'] = oracle_acc


def _run_oracle_episode(model, episode, device):
    """
    Run one oracle-routed episode and return stats dict.
    Same as SRMoETrainer.train_episode but WITHOUT the backward pass.
    Returns the loss as a tensor so it can be combined with L_dagger.
    """
    m = model
    m.reset_episode()
    m.prime_oracle_schedule(list(episode.oracle_actions))

    r_logits_list = []
    oracle_labels = []
    answer_logits = []
    answer_targets = []
    skip_cnt = read_cnt = write_cnt = blocked_cnt = 0
    encode_occupancies = []
    encode_saliences = []

    for t in range(len(episode.input_ids)):
        tok = episode.input_ids[t].to(device)
        tgt = episode.target_value_ids[t].item()
        logits, info = m(tok)

        r_logits_list.append(info['routing_logits'])
        oa = episode.oracle_actions[t]
        oracle_labels.append(ORACLE_TO_ACTION_IDX.get(oa, IGNORE_INDEX))

        ai = info['action_idx']
        if ai == 0: skip_cnt += 1
        elif ai == 1: read_cnt += 1
        elif ai == 2: write_cnt += 1
        if info.get('blocked'): blocked_cnt += 1
        if t < QUERY_START:
            encode_occupancies.append(info['occupancy'])
            encode_saliences.append(info['salience_entropy'])

        if tgt != IGNORE_INDEX:
            answer_logits.append(logits)
            answer_targets.append(tgt)

    n_steps = len(episode.input_ids)
    skip_frac = skip_cnt / n_steps
    read_frac = read_cnt / n_steps
    write_frac = write_cnt / n_steps
    blocked_frac = blocked_cnt / max(n_steps, 1)
    avg_occupancy = sum(encode_occupancies) / max(len(encode_occupancies), 1)
    avg_salience = sum(encode_saliences) / max(len(encode_saliences), 1)

    # Route CE
    r_t = torch.stack(r_logits_list)
    r_lbl = torch.tensor(oracle_labels, dtype=torch.long, device=device)
    route_mask = torch.ones_like(r_lbl, dtype=torch.bool)
    for ti, oa in enumerate(oracle_labels):
        if oa == IGNORE_INDEX:
            route_mask[ti] = False
        elif oa == 0 and ti < QUERY_START:
            route_mask[ti] = False
    r_wt = torch.tensor([1.0, 10.0, 15.0], dtype=torch.float, device=device)
    L_route = F.cross_entropy(r_t, r_lbl, weight=r_wt, ignore_index=IGNORE_INDEX)

    # Answer loss
    L_answer = torch.tensor(0.0, device=device)
    accuracy = 0.0
    if answer_logits:
        a_t = torch.stack(answer_logits)
        a_l = torch.tensor(answer_targets, dtype=torch.long, device=device)
        L_answer = F.cross_entropy(a_t, a_l)
        accuracy = (a_t.argmax(-1) == a_l).float().mean().item()

    L_margin = torch.tensor(0.0, device=device)
    if len(r_logits_list) > 0:
        all_r = torch.stack(r_logits_list)
        L_margin = F.margin_ranking_loss(all_r[:, 2], all_r[:, 0],
                                         torch.ones_like(all_r[:, 0]), margin=0.5)

    L_div = torch.tensor(0.0, device=device)
    if hasattr(m.routing, 'diversity_loss'):
        L_div, _ = m.routing.diversity_loss(m._hidden.detach(),
                                             m.budget.budget_remaining_normalised)

    loss = L_answer + L_route + L_margin + L_div

    return {
        'loss': loss, 'answer_loss': L_answer.item(),
        'route_loss': L_route.item(), 'margin_loss': L_margin.item(),
        'div_loss': L_div.item(), 'accuracy': accuracy,
        'skip_frac': skip_frac, 'read_frac': read_frac, 'write_frac': write_frac,
        'blocked_frac': blocked_frac, 'avg_occupancy': avg_occupancy,
        'avg_salience': avg_salience,
    }

print(f'  DAgger collection every {DAGGER_COLLECT_EVERY} steps, batch_size={DAGGER_BATCH_SIZE}')


In [ ]:
import os
import wandb

WANDB_KEY = "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"
os.environ["WANDB_API_KEY"] = WANDB_KEY

try:
    wandb.init(project="srmoe", name="srmoe-run", dir=os.getcwd())
    print(f"Wandb initialized: {wandb.run.url}")
except Exception as e:
    print(f"Wandb init failed: {e}")

DEEPSEED = 42
N_CODES = 16; RING_DIM = 16; N_PREFS = 32
LAMBDA_DIV = 0.1; LAMBDA_MARGIN = 1.0

torch.manual_seed(DEEPSEED)
model = ArchitectureB(
    d_embed=64, hidden_dim=128, budget_B=55, budget_T=190,
    write_threshold=0.5, embed_seed=DEEPSEED,
    use_sr_moe=True, n_codes=N_CODES, ring_dim=RING_DIM, n_prefs=N_PREFS,
    lambda_div=LAMBDA_DIV, tau_start=1.0, tau_end=0.01, anneal_steps=5000
).to(DEVICE)

trainer = SRMoETrainer(model, lr=3e-3, lambda_route=1.0,
    lambda_div=LAMBDA_DIV, lambda_margin=LAMBDA_MARGIN, margin_target=0.5)

print(f'Total params: {sum(p.numel() for p in model.parameters()):,}')
print(f'SR-MoE routing params: {sum(p.numel() for p in model.routing.parameters()):,}')
print(f'Device: {DEVICE}')

In [ ]:
# Training
# Adjust these to control runtime:
#   GPU (A5000/V100): pretrain=2000, imitation=5000, harden=2000  (~15-20 min)
#   CPU: pretrain=200, imitation=500, harden=200  (~10-20 min)

PRETRAIN_EP = 2000
IMITATION_EP = 5000
HARDEN_EP = 2000

print(f'Phase 1: Supervised pre-training ({PRETRAIN_EP} ep)')
t0 = time.time()
for ep in range(PRETRAIN_EP):
    stats = trainer.train_episode(generate_episode(DEEPSEED + ep), use_oracle_routing=True)
    if ep % 200 == 0 or ep == PRETRAIN_EP - 1:
        print(f'  ep {ep:5d}: loss={stats["loss"]:.4f}  acc={stats["accuracy"]:.3f}  '
              f'route_ce={stats["route_loss"]:.4f}  div={stats["div_loss"]:.4f}  '
              f'wr={stats["write_frac"]:.2f}  rd={stats["read_frac"]:.2f}  '
              f'skip={stats["skip_frac"]:.2f}  occ={stats["avg_occupancy"]:.1f}')
print(f'  Phase 1 done in {time.time()-t0:.0f}s\n')

print(f'Phase 2: DAgger ({IMITATION_EP} ep)')
t0 = time.time()
dagger_buffer = DAggerReplayBuffer(max_size=5000)
phase2_ep = 0
collect_every = 3

for ep in range(IMITATION_EP):
    phase2_ep += 1
    beta = _beta_schedule(phase2_ep)
    
    # Collect DAgger trajectory every N steps
    if phase2_ep % collect_every == 0:
        coll_ep = generate_episode(DEEPSEED + ep + 100000)
        coll_stats = dagger_collection_episode(model, coll_ep, dagger_buffer, DEVICE)
        if coll_stats is not None and phase2_ep % 200 == 0:
            print(f'  coll {phase2_ep:5d}: blkd={coll_stats["blocked_frac"]:.2f} '
                  f'buf={coll_stats["buffer_size"]} '
                  f'wr={coll_stats["write_frac"]:.2f} rd={coll_stats["read_frac"]:.2f}')

    # Training: mix DAgger buffer + fresh oracle episodes
    stats = {}
    dagger_train_step(trainer, dagger_buffer, beta, DEVICE, collect_stats=stats)
    
    if phase2_ep % 200 == 0 or phase2_ep == IMITATION_EP:
        print(f'  p2 {phase2_ep:5d}: beta={beta:.2f} loss={stats.get("mixed_loss",0):.4f} '
              f'd_l={stats.get("dagger_loss",0):.4f} o_l={stats.get("oracle_loss",0):.4f} '
              f'buf={stats.get("buffer_size",0)} acc={stats.get("oracle_acc",0):.3f}')

print(f'Phase 2 done in {time.time()-t0:.0f}s\n')

print(f'Phase 3: Hardening ({HARDEN_EP} ep)')
t0 = time.time()
for ep in range(HARDEN_EP):
    stats = trainer.train_episode(generate_episode(DEEPSEED + ep + 20000), use_oracle_routing=False)
    if ep % 200 == 0 or ep == HARDEN_EP - 1:
        print(f'  ep {ep:5d}: loss={stats["loss"]:.4f}  acc={stats["accuracy"]:.3f}  '
              f'div={stats["div_loss"]:.4f}  margin={stats["margin_loss"]:.4f}  '
              f'wr={stats["write_frac"]:.2f}  rd={stats["read_frac"]:.2f}  '
              f'skip={stats["skip_frac"]:.2f}  blkd={stats["blocked_frac"]:.2f}')
print(f'  Phase 3 done in {time.time()-t0:.0f}s\n')

print('=== Final Evaluation ===')
model.eval()
if hasattr(model.routing, 'set_eval_mode'): model.routing.set_eval_mode(True)
eval_eps = [generate_episode(50000 + i) for i in range(50)]
result = trainer.eval_free(eval_eps, hard=True)
print(f'  free_accuracy:          {result["accuracy"]:.3f}')
print(f'  write_margin:           {result["write_margin"]:.3f}')
print(f'  skip_frac:              {result["skip_frac"]:.3f}')
print(f'  read_frac:              {result["read_frac"]:.3f}')
print(f'  write_frac:            {result["write_frac"]:.3f}')
print(f'  avg_encode_occupancy:   {result["avg_encode_occupancy"]:.1f}')
if trainer._wandb_run is not None:
    trainer._wandb_run.log({
        'eval/free_accuracy': result['accuracy'],
        'eval/write_margin': result['write_margin'],
        'eval/skip_frac': result['skip_frac'],
        'eval/read_frac': result['read_frac'],
        'eval/write_frac': result['write_frac'],
        'eval/avg_encode_occupancy': result['avg_encode_occupancy'],
    })

torch.save({'model': model.state_dict(), 'result': result}, '/workspace/srmoe_result.pt')
print('\nSaved to /workspace/srmoe_result.pt')
